In [ ]:
# ── Cell 1: Imports & Parameters ─────────────────────────────────────────────
# Notebook 11f: Synth Only Condition — EuroSAT
# Train on merged DDIM synthetic EuroSAT train+val
# Test on real EuroSAT test set
# Hyperparameters fixed from notebook 11a HP search

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from sklearn.metrics import classification_report
import gc

# Reproducibility
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

# Data
IMAGE_SIZE = 64
CHANNELS   = 3
CLASSES    = 10

CLASS_NAMES = [
    'AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial',
    'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake'
]

# Fixed hyperparameters from notebook 11a HP search
ACTIVATION = 'relu'
OPTIMIZER  = 'adam'   # default lr=0.001
DROPOUT    = 0.25
BATCH_SIZE = 128
EPOCHS     = 50

# Paths
PROJECT_DIR = '/content/drive/MyDrive/thesis_project'  # adjust to your Drive path
SYNTH_PATH  = os.path.join(PROJECT_DIR, 'data', 'synthetic', 'seed_42', 'ddim',
                            'eurosat', 'eurosat_synthetic.npz')
REAL_PATH   = os.path.join(PROJECT_DIR, 'data', 'processed', 'seed_42', 'eurosat.npz')
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results', 'cnn', 'eurosat', 'synth_only_ddim')
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
# ── Cell 2: Data Loading ──────────────────────────────────────────────────────
# Train: merged synthetic train+val
# Test: real test set (untouched until this final evaluation)

synth_data    = np.load(SYNTH_PATH)
X_train_synth = synth_data['X_train']
y_train_synth = synth_data['y_train']
X_val_synth   = synth_data['X_val']
y_val_synth   = synth_data['y_val']

real_data = np.load(REAL_PATH)
X_test    = real_data['X_test']
y_test    = real_data['y_test']

# Merge synthetic train + val
X_train = np.concatenate([X_train_synth, X_val_synth], axis=0)
y_train = np.concatenate([y_train_synth, y_val_synth], axis=0)

print(f'Synth train: {X_train_synth.shape}, val: {X_val_synth.shape}')
print(f'Merged synth train: {X_train.shape}')
print(f'Real test:   {X_test.shape}')

In [ ]:
# ── Cell 3: Preprocessing ─────────────────────────────────────────────────────

def preprocess(X):
    return X.astype('float32') / 255.0

X_train_n = preprocess(X_train)
X_test_n  = preprocess(X_test)

In [ ]:
# ── Cell 4: Model ─────────────────────────────────────────────────────────────
# Architecture from [CITE SECOND ARTICLE]
# Hyperparameters fixed from notebook 11a HP search

def build_model():
    model = models.Sequential([
        layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, CHANNELS)),

        # Block 1
        layers.Conv2D(64, kernel_size=2, padding='same', activation=ACTIVATION),
        layers.MaxPooling2D(pool_size=2),
        layers.Dropout(DROPOUT),

        # Block 2
        layers.Conv2D(64, kernel_size=2, padding='same', activation=ACTIVATION),
        layers.MaxPooling2D(pool_size=2),
        layers.Dropout(DROPOUT),

        # Block 3 — no pooling as per article
        layers.Conv2D(64, kernel_size=2, padding='same', activation=ACTIVATION),
        layers.Dropout(DROPOUT),

        # FC head
        layers.Flatten(),
        layers.Dense(64, activation=ACTIVATION),
        layers.Dropout(DROPOUT),
        layers.Dense(CLASSES, activation='softmax'),
    ], name='cnn_3conv_2fc')
    return model

tf.keras.backend.clear_session()
gc.collect()

model = build_model()
model.compile(
    optimizer=optimizers.Adam(),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

In [ ]:
# ── Cell 5: Training ──────────────────────────────────────────────────────────

history = model.fit(
    X_train_n, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    verbose=1
)

# Save history
pd.DataFrame(history.history).to_csv(
    os.path.join(RESULTS_DIR, 'history.csv'), index_label='epoch'
)

# Training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['loss'], label='train loss')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['accuracy'], label='train accuracy')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Training Curves — Synth Only (DDIM EuroSAT)', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'training_curves.png'),
            dpi=150, bbox_inches='tight')
plt.close()
print('Training complete.')

In [ ]:
# ── Cell 6: Evaluation on Real Test Set ───────────────────────────────────────

test_loss, test_acc = model.evaluate(X_test_n, y_test, verbose=0)
print(f'Test accuracy: {test_acc:.4f}  |  Test loss: {test_loss:.4f}')

y_pred = np.argmax(model.predict(X_test_n, verbose=0), axis=1)

report = classification_report(
    y_test, y_pred,
    target_names=CLASS_NAMES,
    digits=4
)
print('\nClassification Report:')
print(report)

# Save classification report
report_dict = classification_report(
    y_test, y_pred,
    target_names=CLASS_NAMES,
    digits=4,
    output_dict=True
)
pd.DataFrame(report_dict).transpose().to_csv(
    os.path.join(RESULTS_DIR, 'classification_report.csv')
)

# Save summary
pd.DataFrame([{
    'condition':  'synth_only_ddim',
    'dataset':    'eurosat',
    'test_acc':   round(test_acc, 4),
    'test_loss':  round(test_loss, 4),
}]).to_csv(os.path.join(RESULTS_DIR, 'summary.csv'), index=False)

print(f'\nResults saved to: {RESULTS_DIR}')